In [2]:
#Upload and Load Dataset
import pandas as pd

df = pd.read_csv('/content/spam.csv', encoding='latin-1')
print(df)

     Category                                            Message
0         ham  Go until jurong point, crazy.. Available only ...
1         ham                      Ok lar... Joking wif u oni...
2        spam  Free entry in 2 a wkly comp to win FA Cup fina...
3         ham  U dun say so early hor... U c already then say...
4         ham  Nah I don't think he goes to usf, he lives aro...
...       ...                                                ...
5567     spam  This is the 2nd time we have tried 2 contact u...
5568      ham              Will Ã¼ b going to esplanade fr home?
5569      ham  Pity, * was in mood for that. So...any other s...
5570      ham  The guy did some bitching but I acted like i'd...
5571      ham                         Rofl. Its true to its name

[5572 rows x 2 columns]


In [3]:
#Understand the Dataset
print(df.head())
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print(df.info())

  Category                                            Message
0      ham  Go until jurong point, crazy.. Available only ...
1      ham                      Ok lar... Joking wif u oni...
2     spam  Free entry in 2 a wkly comp to win FA Cup fina...
3      ham  U dun say so early hor... U c already then say...
4      ham  Nah I don't think he goes to usf, he lives aro...
Shape: (5572, 2)
Columns: ['Category', 'Message']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   Category  5572 non-null   object
 1   Message   5572 non-null   object
dtypes: object(2)
memory usage: 87.2+ KB
None


In [5]:
#Clean the Data
data = df[['Category', 'Message']].copy()
data.columns = ['label', 'message']

data = data.dropna()
data = data.drop_duplicates()

print(data.shape)
display(data.head())

(5157, 2)


,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [6]:
#Preprocess the Data
data['label'] = data['label'].str.lower().str.strip()
data = data[data['label'].isin(['ham', 'spam'])].copy()

data['target'] = data['label'].map({'ham': 0, 'spam': 1})

print(data['label'].value_counts())

label
ham     4516
spam     641
Name: count, dtype: int64


In [7]:
#Convert Text into Numbers (TF-IDF)
from sklearn.feature_extraction.text import TfidfVectorizer

X = data['message']
y = data['target']

tfidf = TfidfVectorizer()
X = tfidf.fit_transform(X)

print("Feature matrix shape:", X.shape)

Feature matrix shape: (5157, 8745)


In [8]:
#Split the Dataset
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

Training samples: 4125
Testing samples: 1032


In [9]:
#Train Multiple ML Algorithms
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

models = {
    'Naive Bayes': MultinomialNB(),
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'SVM': LinearSVC(),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42)
}

for name, model in models.items():
    model.fit(X_train, y_train)
    print(name, "trained successfully")

Naive Bayes trained successfully
Logistic Regression trained successfully
SVM trained successfully
Decision Tree trained successfully
Random Forest trained successfully


In [10]:
#Evaluate and Compare Models
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

results = []

for name, model in models.items():
    pred = model.predict(X_test)

    results.append({
        'Algorithm': name,
        'Accuracy': round(accuracy_score(y_test, pred) * 100, 2),
        'Precision': round(precision_score(y_test, pred, zero_division=0) * 100, 2),
        'Recall': round(recall_score(y_test, pred, zero_division=0) * 100, 2),
        'F1 Score': round(f1_score(y_test, pred, zero_division=0) * 100, 2)
    })

results_df = pd.DataFrame(results)
display(results_df.sort_values('F1 Score', ascending=False))

,Algorithm,Accuracy,Precision,Recall,F1 Score
2,SVM,97.48,96.36,82.81,89.08
4,Random Forest,96.90,98.98,75.78,85.84
3,Decision Tree,96.51,88.98,82.03,85.37
1,Logistic Regression,95.83,96.70,68.75,80.37
0,Naive Bayes,95.64,100.00,64.84,78.67


In [11]:
#Predict a New SMS
best_name = results_df.sort_values(
    'F1 Score', ascending=False
).iloc[0]['Algorithm']

best_model = models[best_name]

message = input("Enter an SMS message: ")
message_vector = tfidf.transform([message])
prediction = best_model.predict(message_vector)[0]

print("Best model:", best_name)
print("Prediction:", "SPAM" if prediction == 1 else "HAM (Normal Message)")

Enter an SMS message: good morning
Best model: SVM
Prediction: HAM (Normal Message)
